Vectorize & normalize

In [ ]:
import json
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Load merged feature file
with open("/content/merged_features.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Combine categorical tokens into strings
pos_texts = [" ".join(d["pos"]) for d in data]
dep_texts = [" ".join(d["dep"]) for d in data]
srl_texts = [" ".join(d["srl"]) for d in data]

# Vectorize each type separately
vec_pos = CountVectorizer().fit_transform(pos_texts)
vec_dep = CountVectorizer().fit_transform(dep_texts)
vec_srl = CountVectorizer().fit_transform(srl_texts)

# Combine all feature matrices
import scipy.sparse as sp
X = sp.hstack([vec_pos, vec_dep, vec_srl])

# Convert to dense for PCA (or use TruncatedSVD for large sparse data)
X_dense = X.toarray()
X_scaled = StandardScaler().fit_transform(X_dense)

# Dimensionality reduction (PCA to 2D)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Build DataFrame for easy analysis
df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
df["corpus"] = [d.get("corpus") for d in data]
df["id"] = [d.get("id") for d in data]


Visualize and cluster

In [ ]:
import matplotlib.pyplot as plt

for corpus_name, group in df.groupby("corpus"):
    plt.scatter(group["PC1"], group["PC2"], label=corpus_name, alpha=0.6)

plt.legend()
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("POS+DEP+SRL feature space")
plt.show()
